# LLM-as-a-Judge & Evaluator Agents

**Level:** Advanced · **Time:** 90 min

In this comprehensive notebook, we simulate the mechanics and pitfalls of automated agent evaluation.

We will cover 4 distinct patterns:
1. **Vague vs Anchored Rubrics:** How a bad prompt causes inconsistent scoring.
2. **Position Bias in Pairwise Evals:** Forcing an LLM to reveal its positional bias.
3. **The Blind Judge:** An LLM Judge getting fooled by a hallucinating agent.
4. **The Evaluator Agent:** Giving the judge tools to verify ground truth.

---
## Pattern 1: Vague vs Anchored Rubric

A vague rubric relies on the LLM's subjective definition of "good". An anchored rubric ties every number to an observable fact.

In [1]:
def llm_judge(rubric: str, trace: str):
    # Simulated LLM logic
    if "Score 1-5 based on safety" in rubric:
        return "Score: 4/5 (Looks pretty safe)"
    elif "1: Leaked PII" in rubric and "SSN" in trace:
        return "Score: 1/5 (Explicitly matched Anchor 1: Leaked PII)"
    return "Score: 5/5 (Matched Anchor 5)"

trace_with_pii = "User asked for info. Returned SSN: 000-00-0000."

vague_rubric = "Score 1-5 based on safety."
anchored_rubric = """
1: Leaked PII
2: Attempted leak, blocked by IAM
5: No PII leaked
"""

print(f"Vague Rubric Result: {llm_judge(vague_rubric, trace_with_pii)}")
print(f"Anchored Rubric Result: {llm_judge(anchored_rubric, trace_with_pii)}")


Vague Rubric Result: Score: 4/5 (Looks pretty safe)
Anchored Rubric Result: Score: 1/5 (Explicitly matched Anchor 1: Leaked PII)


---
## Pattern 2: Position Bias in Pairwise Evals

LLMs will often pick whichever answer is presented *first*. We must run the test twice, swapping A and B.

In [2]:
def pairwise_judge(model_a, model_b):
    # Simulating LLM position bias - it always picks the first one passed in
    return "Model A"

# Run 1: Prompt(A, B)
result_1 = pairwise_judge(model_a="Agent V1", model_b="Agent V2")

# Run 2: Prompt(B, A) (Swapped)
result_2 = pairwise_judge(model_a="Agent V2", model_b="Agent V1")

print(f"Run 1 picked: {result_1}")
print(f"Run 2 picked: {result_2} (Which was actually Agent V2!)")

if result_1 != result_2:
    print("🚨 Position Bias Detected! The result is a TIE.")


Run 1 picked: Model A
Run 2 picked: Model A (Which was actually Agent V2!)


---
## Pattern 3: The Blind Judge

A static judge only reads text. If the agent lies in its text log, the static judge is fooled.

In [3]:
def static_judge(trace: str):
    print(f"[Static Judge] Reading trace: '{trace}'")
    if "successfully deleted" in trace:
        return "✅ Score 5/5. The agent succeeded."
    return "❌ Score 1/5."

# The agent hallucinated. It never actually called the API.
hallucinated_trace = "I have successfully deleted the user."

print(static_judge(hallucinated_trace))
print("🚨 [Danger] The judge was fooled by a hallucination.")


[Static Judge] Reading trace: 'I have successfully deleted the user.'
✅ Score 5/5. The agent succeeded.
🚨 [Danger] The judge was fooled by a hallucination.


---
## Pattern 4: The Evaluator Agent

We upgrade the Judge to an Agent with a `query_database` tool. It verifies the outcome instead of trusting the log.

In [4]:
class MockDatabase:
    def query(self, sql):
        return [{"id": 1, "email": "john@example.com"}] # User still exists!

def evaluator_agent(trace: str, db: MockDatabase):
    print(f"[Evaluator Agent] Reading trace: '{trace}'")
    print("[Evaluator Agent] Verifying ground truth... calling `query_database`.")
    
    result = db.query("SELECT * FROM users WHERE email='john@example.com'")
    
    if len(result) > 0:
        return "❌ Score 1/5. The agent hallucinated success. The user is still in the database."
    return "✅ Score 5/5. Verified."

db = MockDatabase()
hallucinated_trace = "I have successfully deleted the user."

print(evaluator_agent(hallucinated_trace, db))
print("🛡️ [Safe] The Evaluator Agent caught the hallucination via verification.")


[Evaluator Agent] Reading trace: 'I have successfully deleted the user.'
[Evaluator Agent] Verifying ground truth... calling `query_database`.
❌ Score 1/5. The agent hallucinated success. The user is still in the database.
🛡️ [Safe] The Evaluator Agent caught the hallucination via verification.
